In [1]:
from IPython.display import display,Markdown
display(Markdown("# Imports"))

# Imports

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from pydantic import BaseModel,Field
from pydantic import Field
from typing import TypedDict,List,Literal
from langchain_core.documents import Document
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
import json
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os

In [ ]:
from IPython.display import display,Markdown
display(Markdown("# Document splitter and Vector Store"))

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

In [ ]:
data={}
with open('../data/articles.json','r') as f:
    data=json.load(f)

In [ ]:
documents=[]
for key,value in zip(data.keys(),data.values()):
    documents.append(Document(metadata={"Article":key},page_content=f"{key} \n: {value}"))

In [ ]:
# vector_store=FAISS.from_documents(documents,embedding=embeddings)

In [ ]:
# vector_store.save_local("../data/constitution.faiss")

In [ ]:
vector_store=FAISS.load_local("../data/constitution.faiss",embeddings=embeddings,allow_dangerous_deserialization=True)

In [ ]:
data={}
with open('../data/penal_code_sections.json','r') as f:
    data=json.load(f)

In [ ]:
documents=[]
for key,value in data.items():
    documents.append(Document(metadata={"Section":key},page_content=f"{key} \n{value}"))

In [ ]:
print(documents[0].page_content)

In [ ]:
# vector_store.add_documents(documents=documents)

In [ ]:
# vector_store.save_local("../data/constitution_and_ipc.faiss")

In [ ]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [ ]:
res=retriever.invoke("murder related law",k=3)
for i in res:
    print(i.page_content)
    print("\n\n")

In [ ]:
from IPython.display import display,Markdown
display(Markdown("# Building nodes and states"))

In [ ]:
class schema(TypedDict):
    retrieval_required:bool
    user_query:str
    k:int=Field(default=3)
    max_retry_for_revise_answer:int=Field(default=3)
    max_retry_for_rewrite_query:int=Field(default=2)
    retrieved_contexts:List[str]
    answer_for_query:str
    relevent_contexts:List[str]
    generated_response:str
    is_grounded:bool
    is_supported:bool
    is_answer_useful:bool
    evidence:str
    retriever

In [ ]:
class schema_for_retrieval_decider_node(BaseModel):
    retrieval_required:bool=Field(...,description="True if retrieval required false otherwise")

parser_for_retrieval_decider_node=PydanticOutputParser(pydantic_object=schema_for_retrieval_decider_node)

sys_prompt_for_retrieval_decider_node=f"You are an AI Assistant . In retriever Indian Penal Code and Consititution of India is present . User will give you some query your task is to determine whether retrieval is required for this query or not \n Output Format -{parser_for_retrieval_decider_node.get_format_instructions()}"


def retrieval_decider_node(state:schema):
    inp=[SystemMessage(content=sys_prompt_for_retrieval_decider_node),HumanMessage(content=f"User Query - {state['user_query']}")]
    res=parser_for_retrieval_decider_node.invoke(main_model.invoke(inp).content).retrieval_required
    return {'retrieval_required':res}

In [ ]:
def retrieve_node(state:schema):
    retrieved_contexts=state['retriever'].invoke(state['user_query'],k=state['k'])
    return {'retrieved_contexts':[i.page_content for i in retrieved_contexts]}

In [ ]:
def direct_generation_node(state:schema):
    res=main_model.invoke(state['user_query']).content
    return {"answer_for_query":res}

In [ ]:
class schema_for_is_relevant_node(BaseModel):
    is_relevant_context:bool
parser_for_is_relevant_node=PydanticOutputParser(pydantic_object=schema_for_is_relevant_node)

def is_relevant_node(state:schema):
    contexts=state['retrieved_context']
    sys_prompt=SystemMessage(content=f"""User will feed you a query  and a retrieved context for that query your task is to tell whether retrieved context is relevent for answering the given query or not \n Output format - {parser_for_is_relevant_node.get_format_instructions()}""")
    hmn_prompt=f"Query - {state['user_query']}"
    lst=[]
    for context in contexts:
        hmn_prompt_dash=hmn_prompt+f"\n Context - {context}"
        res=parser_for_is_relevant_node.invoke(main_model.invoke([sys_prompt,HumanMessage(content=hmn_prompt_dash)]).content)

        lst.append(res.is_relevant_context)

    return {'relevent_contexts':contexts[i] for i in range(len(contexts)) if lst[i]}

In [ ]:
class schema_for_answer_from_context_node(BaseModel):
    response:str=Field(...,description="Response for given query")
    
parser_for_answer_from_context_node=PydanticOutputParser(pydantic_object=schema_for_answer_from_context_node)

def answer_from_context_node(state:schema):
    contexts=state['relevent_contexts']
    sys_prompt_for_answer_from_context_node=SystemMessage(content=f"""User will give you a query and context for query your work is to give answer user query based on the context provided. \n Output format - {parser_for_answer_from_context_node.get_format_instructions()}""")
    
    context=""
    for i in contexts:
        context+=i
        context+="\n"
        
    hmn_prompt=f"Query - {state['user_query']} \n\n Contexts - {context}"
    inp=[SystemMessage(content=sys_prompt_for_answer_from_context_node),HumanMessage(content=hmn_prompt)]

    res=parser_for_answer_from_context_node.invoke(main_model.invoke(inp).content).response
    return {"generated_response":res}

In [ ]:
class schema_for_check_answer_grounded_node(BaseModel):
    is_grounded:Literal['fully_supported','not_fully_supported']
    evidence:str=Field(...,description="Proof that answer is not supported by given contexts")

parser_for_schema_for_check_answer_grounded_node=PydanticOutputParser(pydantic_object=schema_for_check_answer_grounded_node)

def check_answer_grounded_node(state:schema):
    contexts=state['relevent_contexts']
    sys_prompt=SysteMessage(content=f"You are verifying whether the ANSWER is supported by the CONTEXT.\n Output format - {parser_for_support_check_node.get_format_instructions()}")
    context=""
    for i in contexts:
        context+=i
        context+="\n"

    human_pr=HumanMessage(content=f"Answer - {state['generated_response']} \n Contexts - {x}")

    res=parser_for_schema_for_check_answer_grounded_node.invoke(main_model.invoke([sys_prompt,human_pr]).content)
    
    return {'is_grounded':res.is_grounded,'evidence':res.evidence}

In [ ]:
class schema_for_revise_answer_node(BaseModel):
    revised_response:str=Field(...,description="Response for given query")
    
parser_for_revise_answer_node=PydanticOutputParser(pydantic_object=schema_for_revise_answer_node)


def revise_answer_node(state:schema):
    contexts=state['relevent_contexts']
    context=""
    for i in contexts:
        context+=i
        context+="\n"
    generated_response=state['generated_response']
    user_query=state['user_query']
    evidence=state['evidence']
    
    sys_prompt=SystemMessage(content=f"User will give you a query ,an answer for the given query, context for the given query, response generated by llm and a evidence that the response is not fully supported by the given contexts Your task is to revise the answer such that revised answer is fully suported by the given contexts\n Output  format- {parser_for_revise_answer_node.get_format_instructions()}")
    
    
    human_pr=HumanMessage(content=f"Query - {user_query} \n\n Generated Response - {generated_response} \n\n Contexts - {x} \n\n Evidence - {evidence}")

    revised_answer=parser_for_revise_answer_node.invoke(critic_model.invoke([sys_prompt,human_pr]).content)
    
    return {'generated_response':res.revised_response,'max_retry_for_revise_answer':state['max_retry_for_revise_answer']-1}

In [ ]:
class schema_for_is_answer_useful_node(BaseModel):
    is_useful:str=Field(...,description="Boolean response whether answer is useful or not")

parser_for_is_answer_useful_node=PydanticOutputParser(pydantic_object=schema_for_is_answer_useful_node)

def is_answer_useful_node(state:schema):
    user_query=state['user_query']
    generated_response=state['generated_response']

    sys_prompt=SystemMessage(content=f"User will give you a query and a response of the query generated by llm your task is to tell whether the response solves users query or not. Output format - {parser_for_is_answer_useful_node.get_format_instructions()}")
    human_pr=HumanMessage(content=f"Query - {user_query} \n\n Generated Response - {generated_response}")

    res=parser_for_is_answer_useful_node.invoke(judge_model.invoke([sys_prompt,human_pr]).content)
    return {'is_answer_useful':res.is_useful}

In [ ]:
class schema_for_rewrite_query_node(BaseModel):
    updated_query:str

parser_for_rewrite_query_node=PydanticOutputParser(pydantic_object=schema_for_rewrite_query_node)

def rewrite_query_node(state:schema):
    sys_prompt=SystemMessage(content=f"You are a assistant whose work is to modify user query. Our vector database contains documents related to indian constitution and other acts . Users query is too vague to retrieve relevent context to answer the query your task is to rewrite the query such that it is optimised for retrieving correct context. \n Output Format - {parser_for_rewrite_query_node.get_format_instructions()}")
    human_pr=state['user_query']

    res=parser_for_rewrite_query_node.invoke(query_rewrite_model.invoke([sys_prompt,human_pr]).content)
    return {'user_query':res.updated_query,'max_retry_for_rewrite_query':state['max_retry_for_rewrite_query']-1}

In [ ]:
from IPython.display import display,Markdown
display(Markdown("# Building Graph"))

In [ ]:
def retrieval_decider_condition(state:schema):
    return state['retrieval_required']

In [ ]:
def is_relevant_condition(state:schema):
    return state['relevent_contexts']>0

In [ ]:
def is_grounded_condition(state:schema):
    return state['is_grounded'] and state['max_retry_for_revise_answer']>0

In [ ]:
def is_answer_useful_condition(state:schema):
    return state['is_answer_useful'] and state['max_retry_for_rewrite_query']>0

In [ ]:
graph=StateGraph(state_schema=schema)

# adding nodes
# TODO - add a websearch node if none of generated contexts are relevant 
graph.add_node('retrieval_decider_node',retrieval_decider_node)
graph.add_node('retrieve_node',retrieve_node)
graph.add_node('direct_generation_node',direct_generation_node)
graph.add_node('is_relevant_node',is_relevant_node)
graph.add_node('answer_from_context_node',answer_from_context_node)
graph.add_node('check_answer_grounded_node',check_answer_grounded_node)
graph.add_node('revise_answer_node',revise_answer_node)
graph.add_node('is_answer_useful_node',is_answer_useful_node)
graph.add_node('rewrite_query_node',rewrite_query_node)


graph.add_edge(START,'retrieval_decider_node')
graph.add_conditional_edges('retrieval_decider_node',retrieval_decider_condition,{True:'retrieve_node',False:'direct_generation_node'})
graph.add_edge('direct_generation_node',END)
graph.add_edge('retrieve_node','is_relevant_node')
graph.add_conditional_edges('is_relevant_node',is_relevant_condition,{True:'answer_from_context_node',False:END})
graph.add_edge('answer_from_context_node','check_answer_grounded_node')
graph.add_conditional_edges('check_answer_grounded_node',is_grounded_condition,{True:'is_answer_useful_node',False:'revise_answer_node'})
graph.add_edge('revise_answer_node','is_answer_useful_node')
graph.add_conditional_edges('is_answer_useful_node',is_relevant_node,{True:END,False:'rewrite_query_node'})
graph.add_edge('rewrite_query_node','retrieval_decider_node')

workflow=graph.compile()
workflow

In [ ]:
from IPython.display import display,Markdown
display(Markdown("# Adding models"))

In [ ]:
query_rewrite_model=ChatOllama(model='gemma4:31b-cloud')
judge_model=ChatOllama(model='gemma4:31b-cloud')
critic_model=ChatOllama(model='gemma4:31b-cloud')
main_model=ChatOllama(model='gemma4:31b-cloud')

In [ ]:
from IPython.display import display,Markdown
display(Markdown("# Testing workflow"))

In [ ]:
initial_state={
    'retriever':retriever,
    'user_query':'Describe murder releated laws in detail.'
}

In [ ]:
ChatOllama.li

In [ ]:
print(main_model.invoke("what is this?").content)

In [ ]:
response=workflow.invoke(initial_state)